In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Generate Synthetic Time-Series Retail Data
np.random.seed(42)
date_range = pd.date_range(start='2024-01-01', end='2025-12-31', freq='D')
n_days = len(date_range)

# Create a trend + seasonal pattern + noise base
trend = np.linspace(100, 250, n_days)
seasonality = np.sin(date_range.dayofyear * (2 * np.pi / 365.25)) * 40
noise = np.random.normal(0, 15, n_days)
sales = trend + seasonality + noise

df = pd.DataFrame({'Date': date_range, 'Sales': sales})
df.set_index('Date', inplace=True)

# 2. Advanced Feature Engineering for Time Series
df['Year'] = df.index.year
df['Month'] = df.index.month
df['DayOfWeek'] = df.index.dayofweek
df['DayOfYear'] = df.index.dayofyear

# Create Lag features (Looking back at previous days)
df['Sales_Lag_1'] = df['Sales'].shift(1)
df['Sales_Lag_7'] = df['Sales'].shift(7)

# Create Rolling Window features (Moving Average)
df['Sales_Rolling_Mean_7'] = df['Sales'].shift(1).rolling(window=7).mean()

# Drop rows with NaN values generated due to lags/rolling windows
df.dropna(inplace=True)

# 3. Time-Based Train-Test Split (Never split time-series randomly!)
# Use 2024 for training, 2025 for testing
train_df = df[df.index < '2025-01-01']
test_df = df[df.index >= '2025-01-01']

features = ['Year', 'Month', 'DayOfWeek', 'DayOfYear', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Rolling_Mean_7']
target = 'Sales'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

# 4. Model Training using Gradient Boosting Regressor
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

# 5. Inference & Evaluation
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=== Demand Forecasting Metrics (Out-of-Sample Test) ===")
print(f"Mean Absolute Error (MAE): {mae:.2f} units")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} units")

=== Demand Forecasting Metrics (Out-of-Sample Test) ===
Mean Absolute Error (MAE): 60.47 units
Root Mean Squared Error (RMSE): 63.83 units
